# MSF NLP 3 - Group models (run after MSF notebook 2)

**What it does.** It reads the tables that the MSF notebooks made (run cbc18e: 3,912 passages from 893 stories, 9 themes, 15 sub-themes)
and computes every number of the MSF web page. The unit is the **story** (one story = all language versions of one MSF report);
a story raises a theme when any of its passages belongs to it. Ranges resample whole stories; tests shuffle labels between stories.

Parts: (1) themes; (2) countries (8) with a one-country check; (3) country against all others; (4) years 2023 -> 2025 as ordered steps;
(5) the 12 medical topics by country, as reported by notebook 1; (6) themes and medical topics together; (7) one model with country,
year and story length; (8) smallest detectable differences. At the end a zip of the outputs downloads.

**How to run.** Runtime > Run all. No GPU needed. It asks once for Google Drive. Nothing to change.

## Setup: find the MSF tables and check them

In [1]:
import os, glob, csv, json, re, math, collections, datetime, hashlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
csv.field_size_limit(10 ** 9)
import sys, time
try:
    import google.colab  # noqa: F401  (only present inside Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
NEED = ["passages_with_themes.csv", "documents.csv", "s08_group_category_main.csv", "s06_lexicon.json"]
if IN_COLAB:
    PATHS = {f: None for f in NEED}
    # 1) files already uploaded to this Colab session (left panel > Files) are used first
    for f in NEED:
        if os.path.exists("/content/" + f): PATHS[f] = "/content/" + f
    # 2) otherwise look on Google Drive, for at most 2 minutes, so a large Drive cannot make the cell hang
    if any(p is None for p in PATHS.values()):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            t0 = time.time(); found = {}
            for root, dirs, fnames in os.walk("/content/drive/MyDrive"):
                for f in NEED:
                    if PATHS[f] is None and f in fnames:
                        p = os.path.join(root, f)
                        if f not in found or os.path.getmtime(p) > os.path.getmtime(found[f]): found[f] = p
                if len(found) == len(NEED) or time.time() - t0 > 120: break
            for f, p in found.items():
                if PATHS[f] is None: PATHS[f] = p
        except Exception as e:
            print("Google Drive could not be searched (" + str(e)[:120] + "); the files will be asked for instead.")
    # 3) anything still missing: choose the files from your computer
    missing = [f for f, p in PATHS.items() if p is None]
    if missing:
        from google.colab import files
        print("Please choose these files from your computer (you can select all four at once):")
        for f in missing: print("   -", f)
        print("They are in the MSF run folder: export_tables (passages_with_themes.csv, documents.csv) and figures (the other two).")
        up = files.upload()
        for f in missing:
            if os.path.exists("/content/" + f): PATHS[f] = "/content/" + f
        still = [f for f in NEED if PATHS[f] is None]
        assert not still, "Still missing: " + ", ".join(still) + ". Run this cell again and choose them."
    OUT_DIR = "/content/MSF_NB3_outputs"
else:
    # On his PC (VS Code or Jupyter): the MSF run copy in the workspace is found by itself; results go to inputs.
    WS = "<workspace>/"
    os.environ.setdefault("MSF_RUN", WS + "_STUDY/MSF_run_2026-09-25_copy")
    os.environ.setdefault("MSF_OUT", WS + "inputs/MSF_NB3_outputs_" + datetime.date.today().isoformat())
    base = os.environ["MSF_RUN"]
    PATHS = {"passages_with_themes.csv": base + "/export_tables/passages_with_themes.csv", "documents.csv": base + "/export_tables/documents.csv",
             "s08_group_category_main.csv": base + "/figures/s08_group_category_main.csv", "s06_lexicon.json": base + "/figures/s06_lexicon.json"}
    OUT_DIR = os.environ["MSF_OUT"]
os.makedirs(OUT_DIR, exist_ok=True)
for f, p in PATHS.items(): print(f"{f:<30} {p}  (sha256 {hashlib.sha256(open(p, 'rb').read()).hexdigest()[:12]})")
PAS = list(csv.DictReader(open(PATHS["passages_with_themes.csv"], encoding="utf-8")))
DOCS = {r["doc_id"]: r for r in csv.DictReader(open(PATHS["documents.csv"], encoding="utf-8"))}
stories = sorted({p["doc_id"] for p in PAS}); si = {d: i for i, d in enumerate(stories)}; NS = len(stories)
print("passages:", len(PAS), "| stories:", NS, "| documents in the table:", len(DOCS))
assert len(PAS) == 3912 and NS == 893, "This is not the MSF run cbc18e (3,912 passages, 893 stories)."
CN = {"PSE": "Palestine", "SDN": "Sudan", "COD": "DR Congo", "SYR": "Syria", "HTI": "Haiti", "UKR": "Ukraine", "MMR": "Myanmar", "YEM": "Yemen"}
COUNTRY = [CN[DOCS[d]["group"]] for d in stories]; YEAR = [int(DOCS[d]["year"]) for d in stories]
SINGLE = np.array([DOCS[d]["multi_country_flag"] != "YES" for d in stories])
NPAS = np.bincount([si[p["doc_id"]] for p in PAS], minlength=NS).astype(float)
print("stories per country:", dict(collections.Counter(COUNTRY))); print("stories per year:", dict(sorted(collections.Counter(YEAR).items())))
print("one-country stories:", int(SINGLE.sum()), "| translations merged: documents", len(DOCS), "-> stories", NS)
rng = np.random.default_rng(20261114); NPERM, NBOOT = 4000, 2000

passages_with_themes.csv       <workspace>/_STUDY/MSF_run_2026-09-25_copy/export_tables/passages_with_themes.csv  (sha256 24078487cab4)
documents.csv                  <workspace>/_STUDY/MSF_run_2026-09-25_copy/export_tables/documents.csv  (sha256 e558a04ac530)
s08_group_category_main.csv    <workspace>/_STUDY/MSF_run_2026-09-25_copy/figures/s08_group_category_main.csv  (sha256 5b7ed1144c0e)
s06_lexicon.json               <workspace>/_STUDY/MSF_run_2026-09-25_copy/figures/s06_lexicon.json  (sha256 8d839bb5a25c)


passages: 3912 | stories: 893 | documents in the table: 1362
stories per country: {'DR Congo': 127, 'Haiti': 54, 'Ukraine': 47, 'Palestine': 315, 'Yemen': 23, 'Syria': 67, 'Sudan': 214, 'Myanmar': 46}
stories per year: {2023: 237, 2024: 329, 2025: 327}
one-country stories: 705 | translations merged: documents 1362 -> stories 893


## Part 1: themes - how many stories raise each theme

In [2]:
def story_matrix(level):
    ids = sorted({int(p["theme_" + level]) for p in PAS}); names = {}
    V = np.zeros((NS, len(ids)), bool)
    for p in PAS:
        t = int(p["theme_" + level]); V[si[p["doc_id"]], ids.index(t)] = True; names[t] = p["theme_" + level + "_name"]
    return ids, [names[t] for t in ids], V
LEV = {}
for level in ("main", "sub"):
    ids, names, V = story_matrix(level); LEV[level] = dict(ids=ids, names=names, V=V)
def boot_share(V, mask=None):
    idx = np.arange(NS) if mask is None else np.where(mask)[0]; est = V[idx].mean(0); bs = np.empty((NBOOT, V.shape[1]))
    for b in range(NBOOT): bs[b] = V[rng.choice(idx, len(idx))].mean(0)
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0); return est, lo, hi
THEMES = {}
for level in ("main", "sub"):
    L_ = LEV[level]; est, lo, hi = boot_share(L_["V"])
    pas_share = [sum(1 for p in PAS if int(p["theme_" + level]) == t) / len(PAS) for t in L_["ids"]]
    THEMES[level] = [dict(id=t, name=n, stories=int(L_["V"][:, j].sum()), pct=round(float(est[j]), 4), lo=round(float(lo[j]), 4), hi=round(float(hi[j]), 4),
                          passage_pct=round(pas_share[j], 4)) for j, (t, n) in enumerate(zip(L_["ids"], L_["names"]))]
print("PART 1 - share of the 893 stories raising each theme (95% range by story resampling); passages share for comparison")
for r in sorted(THEMES["main"], key=lambda r: -r["pct"]):
    print(f"  {r['name'][:46]:<46} {100*r['pct']:5.1f}% [{100*r['lo']:.0f}-{100*r['hi']:.0f}] of stories | {100*r['passage_pct']:5.1f}% of passages")

PART 1 - share of the 893 stories raising each theme (95% range by story resampling); passages share for comparison
  wounded patients and injuries                   58.0% [55-61] of stories |  34.2% of passages
  displacement and access restrictions            44.9% [42-48] of stories |  24.2% of passages
  children's malnutrition and water sanitation    21.6% [19-24] of stories |  10.3% of passages
  disease outbreak response                       12.8% [11-15] of stories |   9.2% of passages
  maternity care complications                     9.3% [8-11] of stories |   4.5% of passages
  psychological trauma symptoms                    8.8% [7-11] of stories |   5.2% of passages
  earthquake affected areas                        7.8% [6-10] of stories |   4.4% of passages
  sexual violence survivors                        6.5% [5-8] of stories |   3.5% of passages
  migrant routes and asylum access                 6.2% [5-8] of stories |   4.6% of passages


## Part 2: countries - which country is reported for what (tests shuffle the country between stories)

In [3]:
def bh(p):
    p = np.asarray(p, float); o = np.argsort(p); q = np.empty_like(p); m = len(p); prev = 1.0
    for rank in range(m - 1, -1, -1):
        i = o[rank]; prev = min(prev, p[i] * m / (rank + 1)); q[i] = prev
    return q
def onehot(lab, groups):
    G = np.zeros((len(lab), len(groups)))
    for i, g in enumerate(lab):
        if g in groups: G[i, groups.index(g)] = 1
    return G
def chi2_cols(G, k):
    n = np.ones(len(k)); gn = G.T @ n; gk = G.T @ k; tot = gn.sum(); col = gk.sum(0)
    E1 = np.outer(gn, col) / tot; E0 = np.outer(gn, tot - col) / tot
    return (((gk - E1) ** 2) / np.maximum(E1, 1e-9) + (((gn[:, None] - gk) - E0) ** 2) / np.maximum(E0, 1e-9)).sum(0)
def adj_resid(G, k):
    n = np.ones(len(k)); gn = G.T @ n; gk = G.T @ k; tot = gn.sum(); col = gk.sum(0); E = np.outer(gn, col) / tot
    V = E * (1 - gn[:, None] / tot) * (1 - col[None, :] / tot); return (gk - E) / np.sqrt(np.maximum(V, 1e-12))
def table(lab, groups, level, keep=None):
    V = LEV[level]["V"].astype(float); idx = np.array([i for i in range(NS) if lab[i] in groups and (keep is None or keep[i])])
    G = onehot([lab[i] for i in idx], groups); k = V[idx]; obs = chi2_cols(G, k); res = adj_resid(G, k)
    ge = np.zeros(k.shape[1]); ce = np.zeros(res.shape)
    for _ in range(NPERM):
        Gp = G[rng.permutation(len(idx))]; ge += chi2_cols(Gp, k) >= obs - 1e-9; ce += np.abs(adj_resid(Gp, k)) >= np.abs(res) - 1e-9
    p = (ge + 1) / (NPERM + 1); pc = (ce + 1) / (NPERM + 1); q = bh(p); qc = bh(pc.ravel()).reshape(pc.shape)
    gn = G.sum(0); share = (G.T @ k) / gn[:, None]; lo = np.zeros(share.shape); hi = np.zeros(share.shape)
    for g in range(len(groups)):
        mem = np.where(G[:, g] == 1)[0]; bs = np.array([k[rng.choice(mem, len(mem))].mean(0) for _ in range(NBOOT)])
        lo[g], hi[g] = np.percentile(bs, [2.5, 97.5], axis=0)
    return dict(groups=groups, stories=[int(x) for x in gn], themes=LEV[level]["names"], share=share.round(4).tolist(), lo=lo.round(4).tolist(), hi=hi.round(4).tolist(),
                all=k.mean(0).round(4).tolist(), p=p.round(4).tolist(), q=q.round(4).tolist(), resid=res.round(2).tolist(), cell_q=qc.round(4).tolist())
CGROUPS = [c for c, _ in collections.Counter(COUNTRY).most_common()]
GROUPS = {}
for level in ("main", "sub"):
    GROUPS.setdefault("country", {})[level] = table(COUNTRY, CGROUPS, level)
    GROUPS.setdefault("country_single", {})[level] = table(COUNTRY, CGROUPS, level, keep=SINGLE)
print("PART 2 - themes that differ between countries after correction:")
for key in ("country", "country_single"):
    for level in ("main", "sub"):
        t = GROUPS[key][level]; print(f"  {key:<15} {level:<5} {sum(q < .05 for q in t['q'])} of {len(t['q'])}")
lost = [n for n, q1, q2 in zip(GROUPS["country"]["main"]["themes"], GROUPS["country"]["main"]["q"], GROUPS["country_single"]["main"]["q"]) if q1 < .05 and q2 >= .05]
print("  themes that no longer differ with one-country stories only:", lost or "none")

PART 2 - themes that differ between countries after correction:
  country         main  9 of 9
  country         sub   15 of 15
  country_single  main  8 of 9
  country_single  sub   13 of 15
  themes that no longer differ with one-country stories only: ['migrant routes and asylum access']


## Part 3: each country against all the others (percentage points, 95% range)

In [4]:
def contrast(name, A, B_, level):
    V = LEV[level]["V"].astype(float); ia = np.where(A)[0]; ib = np.where(B_)[0]
    e = V[ia].mean(0) - V[ib].mean(0); bs = np.array([V[rng.choice(ia, len(ia))].mean(0) - V[rng.choice(ib, len(ib))].mean(0) for _ in range(NBOOT)])
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0); q = GROUPS["country"][level]["q"]
    return dict(name=name, a_stories=len(ia), b_stories=len(ib), a_pct=(100 * V[ia].mean(0)).round(1).tolist(), b_pct=(100 * V[ib].mean(0)).round(1).tolist(),
                pp=(100 * e).round(1).tolist(), lo=(100 * lo).round(1).tolist(), hi=(100 * hi).round(1).tolist(),
                sig=[bool(q[j] < .05 and (lo[j] > 0 or hi[j] < 0)) for j in range(len(e))])
CON = {level: [contrast(c + " vs all other countries", np.array([x == c for x in COUNTRY]), np.array([x != c for x in COUNTRY]), level) for c in CGROUPS] for level in ("main", "sub")}
print("PART 3 - largest holding difference per country (main themes):")
for c in CON["main"]:
    j = int(np.argmax([abs(v) if s else 0 for v, s in zip(c["pp"], c["sig"])])); print(f"  {c['name'][:32]:<32} {LEV['main']['names'][j][:40]:<40} {c['pp'][j]:+.1f} points [{c['lo'][j]:+.1f}, {c['hi'][j]:+.1f}]" if c["sig"][j] else f"  {c['name'][:32]:<32} no holding difference")

PART 3 - largest holding difference per country (main themes):
  Palestine vs all other countries displacement and access restrictions     +30.7 points [+24.2, +37.3]
  Sudan vs all other countries     children's malnutrition and water sanita +34.3 points [+27.3, +41.3]
  DR Congo vs all other countries  disease outbreak response                +30.1 points [+21.8, +39.2]
  Syria vs all other countries     earthquake affected areas                +39.9 points [+28.0, +52.5]
  Haiti vs all other countries     displacement and access restrictions     -37.9 points [-45.8, -28.7]
  Ukraine vs all other countries   psychological trauma symptoms            +42.3 points [+27.4, +56.4]
  Myanmar vs all other countries   wounded patients and injuries            -31.4 points [-44.4, -18.0]
  Yemen vs all other countries     disease outbreak response                +40.4 points [+20.3, +60.7]


## Part 4: years 2023 -> 2024 -> 2025 as ordered steps

In [5]:
YEARS = sorted(set(YEAR)); ystep = np.array([YEARS.index(y) for y in YEAR], float)
TREND = {}
for level in ("main", "sub"):
    GROUPS.setdefault("year", {})[level] = table([str(y) for y in YEAR], [str(y) for y in YEARS], level)
    V = LEV[level]["V"].astype(float); xc = ystep - ystep.mean(); sl = lambda x, M: ((x - x.mean())[:, None] * M).sum(0) / ((x - x.mean()) ** 2).sum()
    b0 = sl(ystep, V); bs = np.array([(lambda r: sl(ystep[r], V[r]))(rng.integers(0, NS, NS)) for _ in range(NBOOT)]); lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    ge = np.zeros(V.shape[1])
    for _ in range(NPERM): ge += np.abs(sl(ystep[rng.permutation(NS)], V)) >= np.abs(b0) - 1e-12
    p = (ge + 1) / (NPERM + 1); q = bh(p)
    TREND[level] = [dict(theme=n, pp_per_year=round(float(100 * b0[j]), 2), lo=round(float(100 * lo[j]), 2), hi=round(float(100 * hi[j]), 2), p=round(float(p[j]), 4), q=round(float(q[j]), 4))
                    for j, n in enumerate(LEV[level]["names"])]
print("PART 4 - change per year (percentage points of stories; story shuffle, corrected):")
for r in TREND["main"]: print(f"  {r['theme'][:46]:<46} {r['pp_per_year']:+5.1f} per year [{r['lo']:+.1f}, {r['hi']:+.1f}] q {r['q']:.3f}")

PART 4 - change per year (percentage points of stories; story shuffle, corrected):
  wounded patients and injuries                   -2.2 per year [-6.1, +1.8] q 0.476
  displacement and access restrictions            +3.4 per year [-0.6, +7.5] q 0.272
  children's malnutrition and water sanitation    +0.3 per year [-3.0, +3.8] q 0.872
  disease outbreak response                       -0.5 per year [-3.2, +2.2] q 0.872
  psychological trauma symptoms                   -1.6 per year [-3.8, +0.5] q 0.383
  migrant routes and asylum access                -2.1 per year [-4.2, -0.1] q 0.184
  maternity care complications                    -1.1 per year [-3.5, +1.3] q 0.498
  earthquake affected areas                       -4.3 per year [-7.0, -1.7] q 0.002
  sexual violence survivors                       +1.8 per year [-0.2, +3.9] q 0.245


## Part 5: the 12 medical topics by country, as reported by notebook 1 (one version per story, one-country stories)

In [6]:
S8 = list(csv.DictReader(open(PATHS["s08_group_category_main.csv"], encoding="utf-8")))
TOPICS = sorted({r["short_label"] for r in S8}, key=lambda c: -sum(int(r["documents_mentioning"]) for r in S8 if r["short_label"] == c))
TOP_TABLE = [dict(country=r["group_label"], topic=r["short_label"], stories=int(r["n_documents"]), mentioning=int(r["documents_mentioning"]),
                  pct=float(r["document_pct"]), lo=float(r["ci_low"]), hi=float(r["ci_high"])) for r in S8]
print("PART 5 - medical topics: rows read from notebook 1:", len(TOP_TABLE), "| topics:", len(TOPICS))

PART 5 - medical topics: rows read from notebook 1: 96 | topics: 12


## Part 6: themes and medical topics together (word lists of notebook 1 re-applied to each story)

In [7]:
LEX = json.load(open(PATHS["s06_lexicon.json"], encoding="utf-8")); SHORT = LEX["short_labels"]
COMP = {SHORT.get(cat, cat): [re.compile(p, re.I) for v in langs.values() for p in v] for cat, langs in LEX["categories"].items()}
TXT = [DOCS[d]["text_clean"] or "" for d in stories]
TM = np.array([[any(p.search(t) for p in COMP[c]) for c in TOPICS] for t in TXT])
chk = collections.Counter()
for i in range(NS):
    if SINGLE[i]:
        for k, c in enumerate(TOPICS):
            if TM[i, k]: chk[(COUNTRY[i], c)] += 1
ref = {(r["country"], r["topic"]): r["mentioning"] for r in TOP_TABLE}
same = sum(1 for kk in ref if ref[kk] == chk.get(kk, 0)); maxdiff = max(abs(ref[kk] - chk.get(kk, 0)) for kk in ref)
print(f"PART 6 - re-applied word lists match notebook 1 in {same} of {len(ref)} country x topic cells (largest difference {maxdiff} story)")
OVER = {}
for level in ("main", "sub"):
    V = LEV[level]["V"]; base = TM.mean(0)
    OVER[level] = dict(themes=LEV[level]["names"], topics=TOPICS, topic_base=base.round(4).tolist(),
                       pct=[[round(float(TM[V[:, j]][:, k].mean()), 4) for k in range(len(TOPICS))] for j in range(V.shape[1])],
                       lift=[[round(float(TM[V[:, j]][:, k].mean() / max(base[k], 1e-9)), 3) for k in range(len(TOPICS))] for j in range(V.shape[1])])
print("  strongest pairs (share of a theme's stories mentioning the topic, x times the overall rate):")
pairs = sorted(((OVER["main"]["lift"][j][k], j, k) for j in range(len(OVER["main"]["themes"])) for k in range(len(TOPICS))), reverse=True)[:6]
for l, j, k in pairs: print(f"   {OVER['main']['themes'][j][:38]:<38} x {TOPICS[k]:<20} {100*OVER['main']['pct'][j][k]:.0f}% (x{l:.1f})")

PART 6 - re-applied word lists match notebook 1 in 89 of 96 country x topic cells (largest difference 2 story)
  strongest pairs (share of a theme's stories mentioning the topic, x times the overall rate):
   sexual violence survivors              x Sexual violence care 84% (x4.8)
   disease outbreak response              x Vaccination          37% (x4.6)
   maternity care complications           x Maternal/newborn     86% (x3.7)
   psychological trauma symptoms          x Mental health        86% (x3.1)
   disease outbreak response              x Infectious disease   95% (x3.1)
   migrant routes and asylum access       x Sexual violence care 51% (x2.9)


## Part 7: country, year and story length in one model (one logistic model per theme, story level)

In [8]:
import statsmodels.api as sm
REF_C = CGROUPS[0]
X = pd.DataFrame({**{c + " vs " + REF_C: [x == c for x in COUNTRY] for c in CGROUPS[1:]}, **{str(y) + " vs " + str(YEARS[0]): [x == y for x in YEAR] for y in YEARS[1:]},
                  "story length (log passages)": np.log(NPAS)}).astype(float)
COMB = []
for level in ("main", "sub"):
    V = LEV[level]["V"].astype(float); res = []
    for j, nm in enumerate(LEV[level]["names"]):
        y = V[:, j]; empty = [c for c in X.columns[:-1] if X[c].sum() > 0 and y[X[c].values == 1].sum() in (0, X[c].sum())]
        Xj = sm.add_constant(X.drop(columns=empty))
        try:
            m = sm.Logit(y, Xj).fit(disp=False, maxiter=200, cov_type="HC1"); est = {f: (m.params[f], m.bse[f]) for f in X.columns if f not in empty}
            ok = all(np.isfinite(se) and 0 < se < 20 and abs(b) < 7 for b, se in est.values())
        except Exception:
            est, ok = {}, False
        res.append((nm, ok, est, empty))
    for f in X.columns:
        cand = [(nm, est[f]) for nm, ok, est, empty in res if ok and f in est]
        from scipy.stats import norm
        ps = [2 * (1 - norm.cdf(abs(b / se))) for nm, (b, se) in cand]; qs = bh(ps) if ps else []; pq = {nm: (p, q) for (nm, _), p, q in zip(cand, ps, qs)}
        for nm, ok, est, empty in res:
            row = dict(level=level, theme=nm, factor=f, fitted=bool(ok and f in est), note="no story in this group raised it" if f in empty else ("" if ok else "model did not fit"))
            if row["fitted"]:
                b, se = est[f]; p, q = pq[nm]; row.update(OR=float(np.exp(b)), lo=float(np.exp(b - 1.96 * se)), hi=float(np.exp(b + 1.96 * se)), p=float(p), q=float(q))
            COMB.append(row)
held = [r for r in COMB if r["level"] == "main" and r["fitted"] and r["q"] < .05]
print(f"PART 7 - reference: {REF_C} and {YEARS[0]}. Main-theme results that hold after correction: {len(held)}")
for r in sorted(held, key=lambda r: r["q"])[:12]: print(f"  {r['theme'][:38]:<38} {r['factor']:<26} OR {r['OR']:.2f} [{r['lo']:.2f}, {r['hi']:.2f}] q {r['q']:.3f}")

PART 7 - reference: Palestine and 2023. Main-theme results that hold after correction: 48
  children's malnutrition and water sani Sudan vs Palestine         OR 7.51 [4.73, 11.95] q 0.000
  psychological trauma symptoms          story length (log passages) OR 7.60 [4.44, 13.02] q 0.000
  displacement and access restrictions   story length (log passages) OR 2.81 [2.12, 3.72] q 0.000
  displacement and access restrictions   DR Congo vs Palestine      OR 0.18 [0.11, 0.29] q 0.000
  disease outbreak response              DR Congo vs Palestine      OR 65.58 [19.74, 217.86] q 0.000
  earthquake affected areas              Syria vs Palestine         OR 61.34 [18.50, 203.33] q 0.000
  psychological trauma symptoms          Ukraine vs Palestine       OR 12.91 [6.10, 27.34] q 0.000
  maternity care complications           story length (log passages) OR 4.81 [3.00, 7.71] q 0.000
  disease outbreak response              Yemen vs Palestine         OR 113.13 [27.78, 460.64] q 0.000
  displacement an

...\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## Part 8: smallest difference each country comparison could reveal (80% power, 5%)

In [9]:
POWER = []
for c in CGROUPS:
    nA = sum(1 for x in COUNTRY if x == c); nB = NS - nA
    for j, nm in enumerate(LEV["main"]["names"]):
        p0 = LEV["main"]["V"][:, j].mean(); POWER.append(dict(country=c, theme=nm, mde_points=round(100 * (1.959964 + 0.841621) * math.sqrt(p0 * (1 - p0) * (1 / nA + 1 / nB)), 1)))
print("PART 8 - smallest detectable difference, country vs all others:", {c: f"{min(r['mde_points'] for r in POWER if r['country'] == c):.0f}-{max(r['mde_points'] for r in POWER if r['country'] == c):.0f} points" for c in CGROUPS})

PART 8 - smallest detectable difference, country vs all others: {'Palestine': '5-10 points', 'Sudan': '5-11 points', 'DR Congo': '6-13 points', 'Syria': '9-18 points', 'Haiti': '10-20 points', 'Ukraine': '10-21 points', 'Myanmar': '10-21 points', 'Yemen': '14-29 points'}


## Download the results

In [10]:
import shutil
OUT = dict(meta=dict(run="cbc18e", unit="story", stories=NS, passages=len(PAS), documents=len(DOCS), single_country_stories=int(SINGLE.sum()), countries=CGROUPS, years=YEARS,
                     nperm=NPERM, nboot=NBOOT, seed=20261114, lexicon_check=dict(same_cells=same, cells=len(ref), largest_difference=maxdiff)),
           themes=THEMES, groups=GROUPS, contrasts=CON, trend=TREND, topics=dict(order=TOPICS, table=TOP_TABLE), overlap=OVER, combined=COMB, power=POWER,
           stories_per_country=dict(collections.Counter(COUNTRY)), stories_per_year={str(k): v for k, v in sorted(collections.Counter(YEAR).items())})
json.dump(OUT, open(OUT_DIR + "/msf-r4.json", "w", encoding="utf-8"), separators=(",", ":"), default=float)
pd.DataFrame(COMB).to_csv(OUT_DIR + "/msf_combined_model.csv", index=False); pd.DataFrame(TOP_TABLE).to_csv(OUT_DIR + "/msf_topics_by_country.csv", index=False)
stamp = datetime.date.today().isoformat()
z = shutil.make_archive("/content/MSF_NB3_outputs_" + stamp if IN_COLAB else OUT_DIR.rstrip("/"), "zip", OUT_DIR)
print("Zip:", z, "| files:", sorted(os.listdir(OUT_DIR)))
if IN_COLAB:
    from google.colab import files; files.download(z)

Zip: <workspace>\inputs\MSF_NB3_outputs_2026-11-09.zip | files: ['msf-r4.json', 'msf_combined_model.csv', 'msf_topics_by_country.csv']
